# Stock Market Analysis — Visualizations & Insights

This notebook analyses the data warehouse built in `01_etl_pipeline.ipynb` and produces:

1. Price history & closing trends
2. Yearly returns ranking
3. Volatility analysis
4. Daily return distribution
5. Sector market-cap breakdown
6. Moving averages (MA30 / MA90)
7. Volume analysis
8. Financial fundamentals (Revenue, ROE, Net Margin)
9. Correlation heatmap
10. Dividend yield ranking

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

DATA_DIR = Path('../data')
OUT_DIR  = Path('../outputs')
OUT_DIR.mkdir(exist_ok=True)

# ── Plotting defaults ──────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})
PALETTE = sns.color_palette('tab10')
print('Setup complete.')

In [ ]:
# ── Load warehouse tables ──────────────────────────────────────────────────
fact   = pd.read_csv(DATA_DIR / 'fact_stock_prices.csv', parse_dates=['Date'])
dates  = pd.read_csv(DATA_DIR / 'dim_date.csv',          parse_dates=['Date'])
comp   = pd.read_csv(DATA_DIR / 'dim_company.csv')
fin    = pd.read_csv(DATA_DIR / 'dim_financials.csv')

# Master joined frame
df = (
    fact
    .merge(dates, on='Date')
    .merge(comp[['Symbol','Name','Sector','MarketCapitalization','Beta',
                 'EPS','PERatio','DividendPerShare']], on='Symbol')
)

SYMBOLS = sorted(fact['Symbol'].unique())
COLORS  = dict(zip(SYMBOLS, PALETTE[:len(SYMBOLS)]))

print(f'fact rows : {len(fact):,}')
print(f'Symbols   : {SYMBOLS}')
print(f'Date range: {fact.Date.min().date()} → {fact.Date.max().date()}')

## 1. Closing Price History

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(18, 12), sharex=False)
axes = axes.flatten()

for i, sym in enumerate(SYMBOLS):
    sub = fact[fact['Symbol'] == sym].sort_values('Date')
    axes[i].plot(sub['Date'], sub['Close'], color=COLORS[sym], linewidth=0.9)
    axes[i].set_title(sym, fontweight='bold')
    axes[i].set_ylabel('Close (USD)')
    axes[i].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    axes[i].grid(axis='y', alpha=0.3)

plt.suptitle('Daily Closing Price History per Stock', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(OUT_DIR / 'price_history_grid.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Yearly Average Closing Price (all symbols)

In [ ]:
yearly = df.groupby(['Symbol','Year'])['Close'].mean().reset_index()

fig, ax = plt.subplots(figsize=(14, 6))
for sym in SYMBOLS:
    sub = yearly[yearly['Symbol'] == sym]
    ax.plot(sub['Year'], sub['Close'], marker='o', label=sym,
            color=COLORS[sym], linewidth=1.8, markersize=4)

ax.set_title('Yearly Average Closing Price', fontsize=14, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Avg Close (USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend(ncol=3, fontsize=9)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / 'yearly_avg_close.png', dpi=150)
plt.show()

## 3. Yearly Return (%)

In [ ]:
# Compute first and last close per (Symbol, Year)
yr = (
    df.sort_values('Date')
    .groupby(['Symbol','Year'])
    .agg(first_close=('Close','first'), last_close=('Close','last'))
    .reset_index()
)
yr['Return_Pct'] = ((yr['last_close'] - yr['first_close']) / yr['first_close'] * 100).round(2)

# Pivot for heatmap
pivot = yr.pivot(index='Year', columns='Symbol', values='Return_Pct')

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(
    pivot, annot=True, fmt='.1f', cmap='RdYlGn', center=0,
    linewidths=0.5, ax=ax, cbar_kws={'label': 'Yearly Return (%)'}
)
ax.set_title('Yearly Stock Returns (%) — Heatmap', fontsize=14, fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Year')
plt.tight_layout()
plt.savefig(OUT_DIR / 'yearly_returns_heatmap.png', dpi=150)
plt.show()

## 4. Daily Return Distribution

In [ ]:
fact['Daily_Return'] = (fact['Close'] - fact['Open']) / fact['Open'] * 100

fig, axes = plt.subplots(3, 3, figsize=(16, 10))
axes = axes.flatten()

for i, sym in enumerate(SYMBOLS):
    sub = fact[fact['Symbol'] == sym]['Daily_Return'].dropna()
    axes[i].hist(sub, bins=80, color=COLORS[sym], alpha=0.75, edgecolor='none')
    axes[i].axvline(0, color='black', linewidth=0.8, linestyle='--')
    axes[i].set_title(f'{sym}  μ={sub.mean():.2f}%  σ={sub.std():.2f}%', fontsize=10)
    axes[i].set_xlabel('Daily Return (%)')
    axes[i].set_ylabel('Days')

plt.suptitle('Distribution of Daily Returns', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_DIR / 'daily_return_distribution.png', dpi=150)
plt.show()

## 5. Volatility — Avg Intraday Range by Stock

In [ ]:
vol = (
    df.assign(Range=lambda x: x['High'] - x['Low'])
    .groupby(['Symbol','Sector'])['Range']
    .mean()
    .reset_index()
    .sort_values('Range', ascending=False)
)

sector_colors = {s: PALETTE[i] for i, s in enumerate(vol['Sector'].unique())}

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(
    vol['Symbol'], vol['Range'],
    color=[sector_colors[s] for s in vol['Sector']]
)
ax.set_xlabel('Avg Daily Price Range USD (High − Low)')
ax.set_title('Volatility: Average Intraday Price Range', fontsize=13, fontweight='bold')

# Legend for sectors
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=s) for s, c in sector_colors.items()]
ax.legend(handles=legend_elements, title='Sector', fontsize=9)
plt.tight_layout()
plt.savefig(OUT_DIR / 'volatility_intraday_range.png', dpi=150)
plt.show()

## 6. 30-Day & 90-Day Moving Averages

In [ ]:
FOCUS_SYMBOLS = ['AAPL', 'MSFT', 'NVDA', 'TSLA']

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for i, sym in enumerate(FOCUS_SYMBOLS):
    sub = fact[fact['Symbol'] == sym].sort_values('Date').copy()
    sub['MA30'] = sub['Close'].rolling(30).mean()
    sub['MA90'] = sub['Close'].rolling(90).mean()

    ax = axes[i]
    ax.plot(sub['Date'], sub['Close'], color='#aaaaaa', linewidth=0.6, label='Close')
    ax.plot(sub['Date'], sub['MA30'],  color='steelblue', linewidth=1.2, label='MA 30')
    ax.plot(sub['Date'], sub['MA90'],  color='darkorange', linewidth=1.5, label='MA 90')
    ax.set_title(sym, fontsize=12, fontweight='bold')
    ax.set_ylabel('Price (USD)')
    ax.legend(fontsize=8)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('30-Day & 90-Day Moving Averages', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_DIR / 'moving_averages.png', dpi=150)
plt.show()

## 7. Volume Analysis

In [ ]:
# Average daily volume per symbol
vol_avg = fact.groupby('Symbol')['Volume'].mean().sort_values(ascending=False) / 1e6

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(vol_avg.index, vol_avg.values,
              color=[COLORS[s] for s in vol_avg.index], edgecolor='white')
ax.bar_label(bars, fmt='%.0fM', padding=3, fontsize=9)
ax.set_ylabel('Avg Daily Volume (Millions)')
ax.set_title('Average Daily Trading Volume per Stock', fontsize=13, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / 'avg_daily_volume.png', dpi=150)
plt.show()

In [ ]:
# Top 10 highest-volume single days
top_vol = fact.nlargest(10, 'Volume')[['Date','Symbol','Volume','Close']].copy()
top_vol['Volume_M'] = (top_vol['Volume'] / 1e6).round(1)
top_vol

## 8. Sector Market-Cap Breakdown

In [ ]:
sector_cap = comp.groupby('Sector')['MarketCapitalization'].sum().sort_values(ascending=False)
sector_cap_B = sector_cap / 1e12  # in trillions

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Bar chart
sector_cap_B.plot(kind='bar', ax=ax1, color=PALETTE[:len(sector_cap_B)], edgecolor='white')
ax1.set_ylabel('Total Market Cap (Trillions USD)')
ax1.set_title('Market Capitalisation by Sector', fontweight='bold')
ax1.set_xlabel('')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.1f}T'))
ax1.tick_params(axis='x', rotation=20)

# Pie chart
ax2.pie(
    sector_cap_B, labels=sector_cap_B.index,
    autopct='%1.1f%%', startangle=140,
    colors=PALETTE[:len(sector_cap_B)]
)
ax2.set_title('Market Cap Share by Sector', fontweight='bold')

plt.tight_layout()
plt.savefig(OUT_DIR / 'sector_market_cap.png', dpi=150)
plt.show()

## 9. Financial Fundamentals

In [ ]:
# Revenue trend per company
fig, ax = plt.subplots(figsize=(14, 6))
for sym in SYMBOLS:
    sub = fin[fin['Symbol'] == sym].sort_values('Year')
    if sub.empty:
        continue
    col = 'Revenue' if 'Revenue' in sub.columns else None
    if col:
        ax.plot(sub['Year'], sub[col] / 1e3, marker='o', label=sym,
                color=COLORS.get(sym, 'gray'), linewidth=1.5, markersize=4)

ax.set_title('Annual Revenue Trend (2009–2023)', fontsize=13, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Revenue (Billions USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.0f}B'))
ax.legend(ncol=3, fontsize=9)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / 'revenue_trend.png', dpi=150)
plt.show()

In [ ]:
# ROE by company (most recent year)
latest_fin = (
    fin.sort_values('Year')
    .groupby('Symbol')
    .last()
    .reset_index()
    .sort_values('ROE', ascending=False)
)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(
    latest_fin['Symbol'], latest_fin['ROE'],
    color=[COLORS.get(s,'gray') for s in latest_fin['Symbol']],
    edgecolor='white'
)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel('Return on Equity (%)')
ax.set_title('Return on Equity — Most Recent Year', fontsize=13, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / 'roe_comparison.png', dpi=150)
plt.show()

In [ ]:
# Net profit margin trend
fig, ax = plt.subplots(figsize=(14, 6))
for sym in SYMBOLS:
    sub = fin[fin['Symbol'] == sym].sort_values('Year')
    if sub.empty or 'Net_Profit_Margin' not in sub.columns:
        continue
    ax.plot(sub['Year'], sub['Net_Profit_Margin'], marker='o', label=sym,
            color=COLORS.get(sym,'gray'), linewidth=1.5, markersize=4)

ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_title('Net Profit Margin Trend (2009–2023)', fontsize=13, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Net Profit Margin (%)')
ax.legend(ncol=3, fontsize=9)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / 'net_profit_margin.png', dpi=150)
plt.show()

## 10. Correlation Heatmap (Annual Close vs. Fundamentals)

In [ ]:
# Merge annual avg close with financials
ann_close = df.groupby(['Symbol','Year'])['Close'].mean().reset_index().rename(columns={'Close':'Avg_Close'})
merged = ann_close.merge(fin[['Symbol','Year','Revenue','Net_Income','ROE','ROA',
                               'Net_Profit_Margin','Debt_Equity_Ratio']],
                         on=['Symbol','Year'], how='inner')

corr_cols = ['Avg_Close','Revenue','Net_Income','ROE','ROA','Net_Profit_Margin','Debt_Equity_Ratio']
corr_matrix = merged[corr_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax, square=True)
ax.set_title('Correlation: Stock Price vs. Financial Metrics', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_DIR / 'correlation_heatmap.png', dpi=150)
plt.show()

## 11. Dividend Yield Ranking

In [ ]:
div = comp[comp['DividendPerShare'] > 0][['Symbol','Name','DividendPerShare','EPS']].copy()
div['Payout_Ratio'] = (div['DividendPerShare'] / div['EPS'].replace(0, np.nan) * 100).round(2)
div = div.sort_values('DividendPerShare', ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(div['Symbol'], div['DividendPerShare'],
              color=[COLORS.get(s,'gray') for s in div['Symbol']], edgecolor='white')
ax.bar_label(bars, fmt='$%.2f', padding=3)
ax.set_ylabel('Dividend Per Share (USD)')
ax.set_title('Dividend Per Share by Company', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / 'dividend_per_share.png', dpi=150)
plt.show()

print(div.to_string(index=False))

## 12. Key Metrics Summary Table

In [ ]:
# Summary: one row per symbol
latest_fin_d = fin.sort_values('Year').groupby('Symbol').last().reset_index()
price_stats = fact.groupby('Symbol').agg(
    avg_close  = ('Close', 'mean'),
    max_close  = ('Close', 'max'),
    min_close  = ('Close', 'min'),
    std_close  = ('Close', 'std'),
    avg_volume = ('Volume', 'mean'),
).reset_index()

summary = price_stats.merge(comp[['Symbol','Name','Sector','MarketCapitalization',
                                   'Beta','EPS','PERatio']], on='Symbol')
summary = summary.merge(latest_fin_d[['Symbol','ROE','Net_Profit_Margin']], on='Symbol', how='left')

summary['Market_Cap_B'] = (summary['MarketCapitalization'] / 1e9).round(1)
summary = summary.drop(columns=['MarketCapitalization'])

for col in ['avg_close','max_close','min_close','std_close']:
    summary[col] = summary[col].round(2)
summary['avg_volume_M'] = (summary['avg_volume'] / 1e6).round(1)
summary = summary.drop(columns=['avg_volume'])

print('=== Portfolio Summary ===')
summary[['Symbol','Name','Sector','Market_Cap_B','avg_close','max_close','min_close',
          'std_close','Beta','EPS','PERatio','ROE','Net_Profit_Margin']].set_index('Symbol')

In [ ]:
print('All charts saved to outputs/')